# Petri-net AI Workflow — Demo

Load `workflow.yaml`, build the Petri net, fire transitions in order, and inspect the JSON logs written to `artifacts/`.

In [ ]:
import sys
from pathlib import Path

# Make the package importable from the notebooks/ subdirectory.
PKG_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))

from orchestrator import build_net, load_workflow, log_firing, marking_summary

spec = load_workflow(PKG_ROOT / "workflow.yaml")
net = build_net(spec)
print("workflow:", spec["name"])
print("initial marking:", marking_summary(net))

## Fire transitions one at a time

At each step we fire the first enabled transition and print the resulting marking.

In [ ]:
artifacts_dir = PKG_ROOT / "artifacts"
step = 0
while net.enabled_transitions():
    name = net.enabled_transitions()[0]
    transition = net.fire_transition(name)
    step += 1
    log_firing(transition, net, step, artifacts_dir)
    print(f"[step {step}] fired {name!r} -> {marking_summary(net)}")

print("final marking:", marking_summary(net))

## Inspect the JSON logs

In [ ]:
import json

for log_path in sorted(artifacts_dir.glob("step_*.json")):
    entry = json.loads(log_path.read_text())
    print(f"{log_path.name}: {entry['transition']} -> {entry['marking_after']}")